# 04 — Planner, Router, and Specialist Orchestration

## Objective

Demonstrate and test the **planning and routing** layer of the investment research system:

**plan → route → run specialist agents → synthesize**

This notebook covers the following project requirements:

| Requirement | Where it is shown |
|---|---|
| Agent plans its research steps for a stock symbol | Section 3 — Planner Agent |
| Routing workflow: direct content to the right specialist | Section 4 — Router Agent |
| Uses tools / datasets dynamically | Section 5 — Specialist Agents |
| Agent coordination | Sections 6–7 — Synthesis and Full Orchestration |

Notebooks `00`–`03` produce the processed datasets this notebook reads. It does **not** repeat ingestion, preprocessing, or the news chain.

The output of `orchestrate_research()` is the handoff to the Evaluator–Optimizer and memory workflow (Section 9).

## 1. Setup

Code for the agents lives in `src/`:

| Module | Role |
|---|---|
| `src/planner.py` | Planner Agent — builds the ordered research plan |
| `src/router.py` | Router Agent — maps each plan step to a specialist |
| `src/news_agent.py` | News specialist — reads `news_research_results.csv` |
| `src/market_agent.py` | Market specialist — reads `market_data_clean.csv` |
| `src/insider_agent.py` | Insider specialist — reads `sec_form4_clean.csv` |
| `src/synthesis_agent.py` | Synthesis specialist — combines evidence into risks and catalysts |
| `src/orchestrator.py` | Coordinates the full planner → router → specialist workflow |

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.insider_agent import analyze_insider_activity
from src.market_agent import analyze_market
from src.news_agent import analyze_news
from src.orchestrator import display_orchestration, orchestrate_research
from src.planner import generate_research_plan
from src.router import route_step
from src.synthesis_agent import synthesize_research

pd.set_option("display.max_colwidth", 120)

print(f"Project root: {PROJECT_ROOT}")

Project root: /content/Investment-Research-Multi-Agent-System


## 2. Check Required Inputs

The specialist agents read processed files written by notebooks `01` and `03`. If any file is missing, run those notebooks first.

In [2]:
PROCESSED = PROJECT_ROOT / "data" / "processed"

REQUIRED_FILES = {
    "news_agent": PROCESSED / "news_research_results.csv",
    "market_agent": PROCESSED / "market_data_clean.csv",
    "insider_agent": PROCESSED / "sec_form4_clean.csv",
}

missing = [
    f"{agent}: {path.relative_to(PROJECT_ROOT)}"
    for agent, path in REQUIRED_FILES.items()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing processed inputs. Run notebooks 00-03 first:\n"
        + "\n".join(missing)
    )

for agent, path in REQUIRED_FILES.items():
    rows = len(pd.read_csv(path))
    print(f"{agent:<14} {path.name:<28} {rows:>6} rows")

news_agent     news_research_results.csv         6 rows
market_agent   market_data_clean.csv           100 rows
insider_agent  sec_form4_clean.csv              29 rows


### Choose the ticker

Uses the first ticker found in the market data. Change `TICKER` to research a different symbol.

In [3]:
available_tickers = sorted(
    pd.read_csv(REQUIRED_FILES["market_agent"])["ticker"]
    .astype(str)
    .str.upper()
    .unique()
)

print(f"Tickers with market data: {available_tickers}")

TICKER = available_tickers[0]
print(f"Researching: {TICKER}")

Tickers with market data: ['AAPL']
Researching: AAPL


## 3. Planner Agent

Given a ticker, the Planner Agent produces an ordered list of research steps. Each step has a `task` type that the Router uses to pick a specialist. The first three steps gather evidence; the last three combine it.

In [4]:
plan = generate_research_plan(TICKER)

print(plan["objective"])
pd.DataFrame(plan["research_steps"])

Conduct structured multi-agent investment research for AAPL.


,step_id,task,description
0,1,news_analysis,Review recent financial news and macro developments relevant to AAPL.
1,2,market_analysis,"Analyze recent price, return, and trading-volume behavior for AAPL."
2,3,insider_analysis,Review recent SEC Form 4 insider transactions for AAPL.
3,4,risk_analysis,"Identify important risks using the available news, market, and insider evidence for AAPL."
4,5,catalyst_analysis,Identify possible catalysts or notable events affecting AAPL.
5,6,final_synthesis,Synthesize the research evidence into a concise investment-research summary for AAPL.


In [5]:
# Tests: the plan is complete and well-formed.
steps = plan["research_steps"]

assert plan["ticker"] == TICKER
assert [s["step_id"] for s in steps] == list(range(1, len(steps) + 1))
assert all({"step_id", "task", "description"} <= s.keys() for s in steps)
assert TICKER in steps[0]["description"]

# Input is normalised: lower-case and padded tickers give the same plan.
assert generate_research_plan(f"  {TICKER.lower()} ") == plan

print(f"Planner tests passed ({len(steps)} steps).")

Planner tests passed (6 steps).


## 4. Router Agent (Routing Workflow)

The Router reads each plan step and directs it to the specialist that owns that kind of evidence, with a reason for the choice:

| Task | Specialist |
|---|---|
| `news_analysis` | `news_agent` |
| `market_analysis` | `market_agent` |
| `insider_analysis` | `insider_agent` |
| `risk_analysis`, `catalyst_analysis`, `final_synthesis` | `synthesis_agent` |

Any task it does not recognise falls back to `synthesis_agent`.

In [6]:
routing_decisions = [route_step(step) for step in plan["research_steps"]]

pd.DataFrame(routing_decisions)[["step_id", "task", "agent", "reason"]]

,step_id,task,agent,reason
0,1,news_analysis,news_agent,This task depends primarily on processed financial-news evidence.
1,2,market_analysis,market_agent,"This task depends primarily on market price, return, and volume evidence."
2,3,insider_analysis,insider_agent,This task depends primarily on SEC Form 4 insider-transaction evidence.
3,4,risk_analysis,synthesis_agent,This task requires combining evidence from multiple sources.
4,5,catalyst_analysis,synthesis_agent,This task requires combining evidence from multiple sources.
5,6,final_synthesis,synthesis_agent,This task requires combining evidence from multiple sources.


In [7]:
# Tests: every step is routed to the expected specialist.
EXPECTED_ROUTES = {
    "news_analysis": "news_agent",
    "market_analysis": "market_agent",
    "insider_analysis": "insider_agent",
    "risk_analysis": "synthesis_agent",
    "catalyst_analysis": "synthesis_agent",
    "final_synthesis": "synthesis_agent",
}

assert len(routing_decisions) == len(plan["research_steps"])

for decision in routing_decisions:
    assert decision["agent"] == EXPECTED_ROUTES[decision["task"]], decision
    assert decision["reason"]

# Edge cases: task names are normalised, unknown tasks fall back.
assert route_step({"step_id": 1, "task": " NEWS_ANALYSIS "})["agent"] == "news_agent"
assert route_step({"step_id": 99, "task": "unknown_task"})["agent"] == "synthesis_agent"
assert route_step({})["agent"] == "synthesis_agent"

print("Router tests passed.")

Router tests passed.


## 5. Specialist Agents (Tool / Dataset Use)

Each specialist loads only the dataset it needs and returns a structured result for the ticker. If a ticker has no rows in a dataset, the agent returns a count of `0` and a message instead of failing.

In [8]:
news_result = analyze_news(TICKER, PROJECT_ROOT)

print(f"Articles: {news_result['article_count']}")
print(f"Categories: {news_result['categories']}")

pd.DataFrame(news_result["research_notes"]).head()

Articles: 6
Categories: {'macro_market': 2, 'analyst_investor': 2, 'product_service': 1, 'management': 1}


,title,category,event,summary,url
0,"Apple launches Apple Pay in India to take on Google, Paytm",product_service,Apple has launched its Apple Pay service in India in partnership with Axis Bank.,"Apple has launched its Apple Pay service in India in partnership with Axis Bank, alongside expanding its iPhone oper...",https://www.thehindubusinessline.com/info-tech/apple-launches-apple-pay-in-india-to-take-on-google-paytm/article7152...
1,A Look at Apple Inc (AAPL) After 2.7% Decline -- GF Value $287.27 vs Price $329.40,macro_market,"Apple Inc (AAPL) shares fell 2.7% on September 29, 2026, to a price of $329.40.","On September 29, 2026, Apple Inc. (AAPL) shares fell 2.7% to close at $329.40. This places the current share price a...",https://www.gurufocus.com/news/9102424/a-look-at-apple-inc-aapl-after-27-decline-gf-value-28727-vs-price-32940
2,AAPL Reiterated by Morgan Stanley -- Price Target Maintained at $360,analyst_investor,Morgan Stanley analyst Erik Woodring reiterated an Overweight rating for Apple Inc. with a price target maintained a...,"On October 23, 2023, Morgan Stanley analyst Erik Woodring reiterated an Overweight rating on Apple Inc. (AAPL) while...",https://www.gurufocus.com/news/9102050/aapl-reiterated-by-morgan-stanley-price-target-maintained-at-360
3,Apple's New CEO John Ternus Implements Changes to Boost Product Development,management,"John Ternus becomes the new CEO of Apple Inc. and implements changes to enhance product development on September 29,...","On September 29, 2026, John Ternus became the new CEO of Apple Inc. and implemented significant changes to enhance p...",https://www.gurufocus.com/news/9101935/apples-new-ceo-john-ternus-implements-changes-to-boost-product-development
4,"Meta's AI Muse Challenges Apple's Ecosystem, Raising Concerns for AAPL",analyst_investor,"On September 29, 2026, Bank of America highlighted emerging risks for Apple Inc. as Meta Platforms' AI-powered agent...","On September 29, 2026, Bank of America highlighted emerging risks for Apple Inc. driven by the rapid popularity of M...",https://www.gurufocus.com/news/9101853/metas-ai-muse-challenges-apples-ecosystem-raising-concerns-for-aapl


In [9]:
market_result = analyze_market(TICKER, PROJECT_ROOT)
market_result

{'agent': 'market_agent',
 'ticker': 'AAPL',
 'row_count': 100,
 'start_date': Timestamp('2026-05-07 00:00:00+0000', tz='UTC'),
 'end_date': Timestamp('2026-09-29 00:00:00+0000', tz='UTC'),
 'latest_close': 329.4,
 'latest_volume': 38478037.0,
 'average_daily_return': 0.0015361584653877881,
 'average_volume': 50982172.3}

In [10]:
insider_result = analyze_insider_activity(TICKER, PROJECT_ROOT)
insider_result

{'agent': 'insider_agent',
 'ticker': 'AAPL',
 'transaction_count': 29,
 'unique_insiders': 8,
 'acquired_transactions': 11,
 'disposed_transactions': 18,
 'total_transaction_value': 101329036.81,
 'start_date': Timestamp('2026-04-23 00:00:00+0000', tz='UTC'),
 'end_date': Timestamp('2026-09-27 00:00:00+0000', tz='UTC')}

In [11]:
# Tests: each specialist reports on the right ticker with the expected fields.
for result, name, count_key in [
    (news_result, "news_agent", "article_count"),
    (market_result, "market_agent", "row_count"),
    (insider_result, "insider_agent", "transaction_count"),
]:
    assert result["agent"] == name
    assert result["ticker"] == TICKER
    assert result[count_key] >= 0

# Specialists handle a ticker with no data without raising.
empty_news = analyze_news("NO_SUCH_TICKER", PROJECT_ROOT)
empty_market = analyze_market("NO_SUCH_TICKER", PROJECT_ROOT)
empty_insider = analyze_insider_activity("NO_SUCH_TICKER", PROJECT_ROOT)

assert empty_news["article_count"] == 0
assert empty_market["row_count"] == 0
assert empty_insider["transaction_count"] == 0

print("Specialist tests passed.")

Specialist tests passed.


## 6. Synthesis Agent

The Synthesis Agent combines the three specialist results into **risks** and **catalysts**, using simple rules over news categories, average daily return, and insider acquired/disposed counts.

In [12]:
synthesis_result = synthesize_research(
    TICKER,
    news_result,
    market_result,
    insider_result,
)

print("Risks:")
for risk in synthesis_result["risks"]:
    print(f"- {risk}")

print("\nCatalysts:")
for catalyst in synthesis_result["catalysts"]:
    print(f"- {catalyst}")

print(f"\nEvidence used: {synthesis_result['summary']}")

Risks:
- Macro or broader market developments may affect the stock.
- Disposed insider transactions outnumber acquired transactions in the available Form 4 data.

Catalysts:
- Recent analyst or investor commentary may influence market expectations.
- Average daily market return over the available period is positive.
- The available Form 4 data includes insider acquisition transactions.

Evidence used: {'news_articles': 6, 'market_rows': 100, 'insider_transactions': 29}


In [13]:
# Tests: synthesis rules fire correctly on controlled inputs.
bearish = synthesize_research(
    "TEST",
    {"categories": {"macro_market": 2}, "article_count": 2},
    {"average_daily_return": -0.01, "row_count": 10},
    {"acquired_transactions": 1, "disposed_transactions": 4, "transaction_count": 5},
)
assert len(bearish["risks"]) == 3
assert bearish["summary"] == {
    "news_articles": 2,
    "market_rows": 10,
    "insider_transactions": 5,
}

bullish = synthesize_research(
    "TEST",
    {"categories": {"analyst_investor": 1}},
    {"average_daily_return": 0.02},
    {"acquired_transactions": 3, "disposed_transactions": 0},
)
assert bullish["risks"] == []
assert len(bullish["catalysts"]) == 3

# Missing evidence produces empty lists, not an error.
empty = synthesize_research("TEST", {}, {}, {})
assert empty["risks"] == [] and empty["catalysts"] == []

print("Synthesis tests passed.")

Synthesis tests passed.


## 7. Full Orchestration

`orchestrate_research()` runs the whole workflow in one call: the Planner builds the plan, the Router assigns each step, the specialists run, and the Synthesis Agent combines their results. Each executed step records which agent handled it and why.

In [14]:
result = orchestrate_research(TICKER, PROJECT_ROOT)

display_orchestration(result)

Ticker: AAPL
Objective: Conduct structured multi-agent investment research for AAPL.
Step 1: news_analysis -> news_agent
Step 2: market_analysis -> market_agent
Step 3: insider_analysis -> insider_agent
Step 4: risk_analysis -> synthesis_agent
Step 5: catalyst_analysis -> synthesis_agent
Step 6: final_synthesis -> synthesis_agent
Final synthesis
Risks:
- Macro or broader market developments may affect the stock.
- Disposed insider transactions outnumber acquired transactions in the available Form 4 data.
Catalysts:
- Recent analyst or investor commentary may influence market expectations.
- Average daily market return over the available period is positive.
- The available Form 4 data includes insider acquisition transactions.


In [15]:
pd.DataFrame(
    [
        {
            "step_id": step["step_id"],
            "task": step["task"],
            "agent": step["agent"],
            "reason": step["reason"],
        }
        for step in result["executed_steps"]
    ]
)

,step_id,task,agent,reason
0,1,news_analysis,news_agent,This task depends primarily on processed financial-news evidence.
1,2,market_analysis,market_agent,"This task depends primarily on market price, return, and volume evidence."
2,3,insider_analysis,insider_agent,This task depends primarily on SEC Form 4 insider-transaction evidence.
3,4,risk_analysis,synthesis_agent,This task requires combining evidence from multiple sources.
4,5,catalyst_analysis,synthesis_agent,This task requires combining evidence from multiple sources.
5,6,final_synthesis,synthesis_agent,This task requires combining evidence from multiple sources.


In [16]:
# Tests: the end-to-end run matches the individual steps above.
assert result["ticker"] == TICKER
assert result["plan"] == plan
assert result["routing_decisions"] == routing_decisions
assert len(result["executed_steps"]) == len(plan["research_steps"])

for step in result["executed_steps"]:
    assert step["result"] is result["specialist_results"][step["agent"]]

assert result["final_synthesis"]["risks"] == synthesis_result["risks"]
assert result["final_synthesis"]["catalysts"] == synthesis_result["catalysts"]

print("Orchestration tests passed.")

Orchestration tests passed.


## 8. Run for Every Available Ticker

Confirms the workflow runs end to end for each ticker in the processed data.

In [17]:
summary_rows = []

for ticker in available_tickers:
    run = orchestrate_research(ticker, PROJECT_ROOT)
    evidence = run["final_synthesis"]["summary"]
    summary_rows.append(
        {
            "ticker": ticker,
            "news_articles": evidence["news_articles"],
            "market_rows": evidence["market_rows"],
            "insider_transactions": evidence["insider_transactions"],
            "risks": len(run["final_synthesis"]["risks"]),
            "catalysts": len(run["final_synthesis"]["catalysts"]),
        }
    )

pd.DataFrame(summary_rows)

,ticker,news_articles,market_rows,insider_transactions,risks,catalysts
0,AAPL,6,100,29,2,3


## 9. Handoff to Evaluator–Optimizer and Memory

The Evaluator–Optimizer workflow should call the orchestrator rather than re-run the agents:

```python
from src.orchestrator import orchestrate_research

result = orchestrate_research(ticker, PROJECT_ROOT)
```

Useful fields in `result`:

| Key | Contents |
|---|---|
| `final_synthesis["risks"]` | List of risk statements |
| `final_synthesis["catalysts"]` | List of catalyst statements |
| `final_synthesis["summary"]` | Evidence counts (news, market, insider) |
| `specialist_results` | Full output of each specialist, for evaluating evidence coverage |
| `executed_steps` | Step-by-step plan, agent, and reason — useful to store as run memory |

This notebook does not implement self-reflection, refinement, or cross-run memory.